In [ ]:
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

X_train = pd.read_csv("data/train.csv")
y_train = X_train.pop("addicted_label")
X_train = X_train.drop("id", axis=1)

cat = ["gender", "stress_level", "academic_work_impact"]

for c in cat:
    X_train[c] = X_train[c].fillna("missing").astype(str)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = []

for train_idx, val_idx in skf.split(X_train, y_train):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    model = CatBoostClassifier(cat_features=cat, verbose=False)
    model.fit(X_tr, y_tr)

    preds = model.predict_proba(X_val)[:, 1]
    scores.append(roc_auc_score(y_val, preds))

scores = np.array(scores)
print(scores)
print(scores.mean())

[0.96223054 0.96282651 0.96291912 0.96359831 0.96236973]
0.9627888433905024


In [9]:
model.fit(X_train, y_train)

X_test = pd.read_csv("data/test.csv")
IDs = X_test.pop("id")

for c in cat:
    X_test[c] = X_test[c].fillna("missing").astype(str)

preds = model.predict_proba(X_test)[:, 1]

sub = pd.DataFrame(data={"addicted_label": preds}, index=IDs)
sub.to_csv("catboost_baseline.csv")